# App Activity Analysis

Given an app activity table `b_sql.b_practice.activity` with columns `user_id`, `event_name`, `event_date`, and `country`, solve the following:

1. **Same-day conversions** — For each date, count how many users installed the app and made a purchase on that same day.
2. **Daily active users** — Count distinct active users per day.
3. **Weekly active users** — Count distinct active users per week of the month.

In [0]:
%sql
-- Create and populate the activity table
CREATE table b_sql.b_practice.activity
(
user_id varchar(20),
event_name varchar(20),
event_date date,
country varchar(20)
);
delete from b_sql.b_practice.activity;
insert into b_sql.b_practice.activity values (1,'app-installed','2022-01-01','India')
,(1,'app-purchase','2022-01-02','India')
,(2,'app-installed','2022-01-01','USA')
,(3,'app-installed','2022-01-01','USA')
,(3,'app-purchase','2022-01-03','USA')
,(4,'app-installed','2022-01-03','India')
,(4,'app-purchase','2022-01-03','India')
,(5,'app-installed','2022-01-03','SL')
,(5,'app-purchase','2022-01-03','SL')
,(6,'app-installed','2022-01-04','Pakistan')
,(6,'app-purchase','2022-01-04','Pakistan');

In [0]:
%sql
-- Preview the activity data
select * from b_sql.b_practice.activity

In [0]:
# Import PySpark functions, types, and Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Load the activity table into a PySpark DataFrame
df=spark.read.table("b_sql.b_practice.activity")
df.display()

In [0]:
# Q1: Same-day install-to-purchase conversions per date
# Use LAG to compare each user's current event with their previous event,
# flag rows where install and purchase happened on the same day.
df_p=(
    df.withColumn("prev_date",
                  lag("event_date",1).over(
                      Window.partitionBy("user_id").orderBy("event_date")
                      )
    ).withColumn("prev_event",
                  lag("event_name",1).over(
                      Window.partitionBy("user_id").orderBy("event_date")
                      )))
df_p_filter=(df_p.withColumn("diff",
                datediff("event_date","prev_date"))
                .withColumn("flag",
                    when((col("diff")==0) & (col("prev_event")=="app-installed") & (col("event_name")=="app-purchase"),1)
                    .otherwise(0))
)
df_p_filter.groupBy("event_date").agg(sum("flag").alias("total")).display()
                


In [0]:
# Q2: Count distinct active users per day
df_a_each_day=df.groupBy("event_date").agg(countDistinct("user_id").alias("total_active_users"))
df_a_each_day.display()

In [0]:

# Q3: Count distinct active users per week of month
df_a_each_week = (
    df
    .withColumn("week", expr("extract(WEEK FROM event_date)"))
    .groupBy("week")
    .agg(countDistinct("user_id").alias("total_active_users"))
)

df_a_each_week.display()

